# 02 - Engenharia de Features

## Passos Mágicos - Predição de Risco de Defasagem Escolar

Este notebook demonstra o processo de engenharia de features para o modelo de predição.

### Objetivos:
1. Criar features derivadas dos indicadores
2. Calcular variável target (RISCO_DEFASAGEM)
3. Selecionar features mais relevantes
4. Preparar dados para modelagem

In [ ]:
# Imports
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import DataLoader, DataCleaner, DataTransformer
from src.features import FeatureEngineer, FeatureSelector
from src.config import RISCO_LABELS

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Carregamento e Limpeza dos Dados

In [ ]:
# Carregar e limpar dados
loader = DataLoader()
df = loader.load_and_unify()

cleaner = DataCleaner()
df_clean = cleaner.fit_transform(df)

print(f"Dados carregados: {df_clean.shape}")
print(f"\nRelatório de limpeza:")
print(cleaner.get_cleaning_report())

## 2. Criação de Features

In [ ]:
# Aplicar engenharia de features
engineer = FeatureEngineer()
df_feat = engineer.create_all_features(df_clean)

print(f"Colunas originais: {len(df_clean.columns)}")
print(f"Colunas após features: {len(df_feat.columns)}")
print(f"\nFeatures criadas: {len(engineer.get_created_features())}")
print(engineer.get_created_features())

## 3. Análise da Variável Target

In [ ]:
# Distribuição do target
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Contagem
target_counts = df_feat['RISCO_DEFASAGEM'].value_counts().sort_index()
colors = ['green', 'orange', 'red']
target_counts.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Distribuição do Risco de Defasagem')
axes[0].set_xticklabels(['BAIXO (0)', 'MÉDIO (1)', 'ALTO (2)'], rotation=0)
axes[0].set_ylabel('Quantidade')

# Percentual
target_pct = (target_counts / len(df_feat) * 100).round(1)
target_pct.plot(kind='pie', ax=axes[1], colors=colors, autopct='%1.1f%%',
                labels=['BAIXO', 'MÉDIO', 'ALTO'])
axes[1].set_title('Percentual por Nível de Risco')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("\nDistribuição detalhada:")
for risk, count in target_counts.items():
    pct = count / len(df_feat) * 100
    print(f"  {RISCO_LABELS[risk]}: {count} ({pct:.1f}%)")

## 4. Seleção de Features

In [ ]:
# Preparar X e y
exclude_cols = ['RISCO_DEFASAGEM', 'NIVEL_RISCO', 'RA', 'NOME', 'ANO_PEDE', 
                'PEDRA', 'GENERO', 'INSTITUICAO_ENSINO_ALUNO', 'TURMA', 'FAIXA_ETARIA', 'INDE_CATEGORIA']

feature_cols = [col for col in df_feat.columns 
                if col not in exclude_cols 
                and df_feat[col].dtype in ['int64', 'float64', 'int32', 'float32']]

X = df_feat[feature_cols].copy()
y = df_feat['RISCO_DEFASAGEM'].copy()

print(f"Features numéricas disponíveis: {len(feature_cols)}")
print(f"Amostras: {len(X)}")

In [ ]:
# Seleção de features por importância
selector = FeatureSelector(method='importance', n_features=15)
X_selected = selector.fit_transform(X, y)

print(f"Features selecionadas: {len(selector.get_selected_features())}")
print("\nTop 15 features:")
for i, feat in enumerate(selector.get_selected_features(), 1):
    print(f"  {i}. {feat}")

In [ ]:
# Visualização das importâncias
importances = selector.get_feature_importances()
top_15 = dict(sorted(importances.items(), key=lambda x: x[1], reverse=True)[:15])

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(x=list(top_15.values()), y=list(top_15.keys()), palette='viridis')
plt.xlabel('Importância')
plt.title('Top 15 Features mais Importantes')
plt.tight_layout()
plt.show()

## 5. Salvar Dados Processados

In [ ]:
# Salvar dados processados
from src.config import PROCESSED_DATA_DIR
from src.utils.helpers import ensure_dir

ensure_dir(PROCESSED_DATA_DIR)

# Salvar features selecionadas + target
df_final = X_selected.copy()
df_final['RISCO_DEFASAGEM'] = y.values

output_path = PROCESSED_DATA_DIR / 'features_selected.parquet'
df_final.to_parquet(output_path, index=False)

print(f"Dados salvos em: {output_path}")
print(f"Shape: {df_final.shape}")

## 6. Conclusões

### Features Criadas:
- **Defasagem**: FASE_IDEAL, DEFASAGEM, TEM_DEFASAGEM
- **Temporais**: ANOS_PM, VETERANO
- **Indicadores**: MEDIA_INDICADORES, STD_INDICADORES, INDICADORES_BAIXOS
- **Interações**: RATIO_IDA_IEG, ENGAJ_X_DESEMP, SCORE_ACADEMICO

### Target (RISCO_DEFASAGEM):
- BAIXO (0): Sem defasagem, INDE >= 6.5
- MÉDIO (1): Defasagem = 1 ou INDE entre 5.5-6.5
- ALTO (2): Defasagem >= 2 ou INDE < 5.5

### Próximos Passos:
- Treinar e comparar modelos
- Otimizar hiperparâmetros
- Avaliar performance